In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

print("=" * 80)
print("📊 STEP 3: Generating orders.csv (100,000 rows with quality issues)...")
print("=" * 80)

# Read back customers and products to get valid IDs
customers_df = pd.read_csv('/Volumes/workspace/dev/medallion_data/customers.csv')
products_df = pd.read_csv('/Volumes/workspace/dev/medallion_data/products.csv')

valid_customer_ids = customers_df['customer_id'].drop_duplicates().tolist()
valid_product_ids = products_df['product_id'].tolist()

print(f"\nValid customer IDs: {len(valid_customer_ids)}")
print(f"Valid product IDs: {len(valid_product_ids)}")

num_orders = 100000
order_ids = list(range(1, num_orders + 1))

# Generate base orders
order_customer_ids = [random.choice(valid_customer_ids) for _ in range(num_orders)]
order_product_ids = [random.choice(valid_product_ids) for _ in range(num_orders)]

# Order dates (2023-2024 for recent activity)
order_start = datetime(2023, 1, 1)
order_end = datetime(2024, 12, 31)
order_date_range = (order_end - order_start).days
order_dates = [order_start + timedelta(days=random.randint(0, order_date_range)) for _ in range(num_orders)]

# Quantities and prices
quantities = [random.randint(1, 10) for _ in range(num_orders)]

# Get prices from products
product_price_map = dict(zip(products_df['product_id'], products_df['price']))
unit_prices = [product_price_map.get(pid, 100.0) for pid in order_product_ids]
total_amounts = [round(q * p, 2) for q, p in zip(quantities, unit_prices)]

# Order statuses
statuses = ['Completed', 'Pending', 'Cancelled']
status_weights = [0.85, 0.10, 0.05]  # 85% completed
order_statuses = random.choices(statuses, weights=status_weights, k=num_orders)

# Payment dates (NULL for pending/cancelled, 1-30 days after order for completed)
payment_dates = []
for i, status in enumerate(order_statuses):
    if status == 'Completed':
        payment_delay = random.randint(0, 30)
        payment_date = order_dates[i] + timedelta(days=payment_delay)
        payment_dates.append(payment_date.strftime('%Y-%m-%d'))
    else:
        payment_dates.append(None)

orders_df = pd.DataFrame({
    'order_id': order_ids,
    'customer_id': order_customer_ids,
    'order_date': [d.strftime('%Y-%m-%d') for d in order_dates],
    'product_id': order_product_ids,
    'quantity': quantities,
    'unit_price': unit_prices,
    'total_amount': total_amounts,
    'order_status': order_statuses,
    'payment_date': payment_dates
})

print("\n🔴 Injecting order quality issues:")

# Issue 1: 100 rows with NULL customer_id
null_customer_indices = random.sample(range(len(orders_df)), 100)
orders_df.loc[null_customer_indices, 'customer_id'] = None
print(f"  ✓ Issue 1: Added 100 NULL customer_id (completeness)")

# Issue 2: 200 rows with NULL product_id  
null_product_indices = random.sample(range(len(orders_df)), 200)
orders_df.loc[null_product_indices, 'product_id'] = None
print(f"  ✓ Issue 2: Added 200 NULL product_id (completeness)")

# Issue 3: 50 rows with invalid customer_id (not in customers table)
invalid_customer_indices = random.sample(range(len(orders_df)), 50)
invalid_customer_ids = list(range(20000, 20050))  # IDs that don't exist
for idx, invalid_id in zip(invalid_customer_indices, invalid_customer_ids):
    orders_df.loc[idx, 'customer_id'] = invalid_id
print(f"  ✓ Issue 3: Added 50 invalid customer_id (referential integrity)")

# Issue 4: 30 rows with invalid product_id (not in products table)
invalid_product_indices = random.sample(range(len(orders_df)), 30)
invalid_product_ids = list(range(10000, 10030))  # IDs that don't exist
for idx, invalid_id in zip(invalid_product_indices, invalid_product_ids):
    orders_df.loc[idx, 'product_id'] = invalid_id
print(f"  ✓ Issue 4: Added 30 invalid product_id (referential integrity)")

# Issue 5: 20 duplicate order_id rows
duplicate_order_indices = random.sample(range(1000, 2000), 20)
duplicate_order_rows = orders_df[orders_df['order_id'].isin(duplicate_order_indices)].copy()
duplicate_order_rows['order_status'] = 'Pending'  # Make them slightly different
orders_df = pd.concat([orders_df, duplicate_order_rows], ignore_index=True)
print(f"  ✓ Issue 5: Added 20 duplicate order_id rows (uniqueness)")

# Shuffle to distribute quality issues
orders_df = orders_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save to Volume
order_path = '/Volumes/workspace/dev/medallion_data/orders.csv'
orders_df.to_csv(order_path, index=False)

print(f"\n✅ Orders.csv generated:")
print(f"   Total rows: {len(orders_df):,}")
print(f"   NULL customer_ids: {orders_df['customer_id'].isna().sum()}")
print(f"   NULL product_ids: {orders_df['product_id'].isna().sum()}")
print(f"   Duplicate order_ids: {orders_df['order_id'].duplicated().sum()}")
print(f"   Saved to: {order_path}")

# Calculate quality issue summary
total_issues = 100 + 200 + 50 + 30 + 20  # 400 issues
print(f"\n📊 Quality Issue Summary:")
print(f"   Total quality issues: {total_issues} rows")
print(f"   Quality issue rate: {(total_issues / 100000) * 100:.2f}%")
print(f"   Clean data rate: {((100000 - total_issues) / 100000) * 100:.2f}%")

print("\nOrders generation complete! ✅")